# Table Relationship Exploration

**Purpose:** Identify join relationships between tables in the bronze layer (`databricks_bootcamp_dwb.bronze`) to guide transformations in the silver layer.

**Approach:** Schema-level analysis only — we examine column names, data types, and metadata to hypothesize relationships, without querying row-level data.

**Workflow:**
1. Import tables in schema order (fixed for entire analysis)
2. Identify candidate join columns
3. Group candidates by semantic field
4. Build hypothesis join diagram
5. (Later) Test hypotheses with actual joins

In [0]:
# List all tables in databricks_bootcamp_dwb.bronze in schema order
tables = spark.sql("SHOW TABLES IN databricks_bootcamp_dwb.bronze").select("tableName").collect()
table_names = [row.tableName for row in tables]

print("Tables in databricks_bootcamp_dwb.bronze (locked order):")
for i, table in enumerate(table_names, 1):
    print(f"{i}. {table}")

# Import each table into a dataframe
table_dfs = {}
for table_name in table_names:
    table_dfs[table_name] = spark.table(f"databricks_bootcamp_dwb.bronze.{table_name}")

print(f"\nImported {len(table_dfs)} tables.")

In [0]:
# Define target silver table mappings based on silver notebook documentation
target_mappings = {
    'crm_cust_info': 'crm_customers',
    'crm_prd_info': 'crm_products',
    'crm_sales_details': 'crm_sales',
    'erp_cust_az12': 'erp_customers',
    'erp_loc_a101': 'erp_customer_location',
    'erp_px_cat_g1v2': 'erp_products'
}

print("Bronze → Silver Target Mappings:")
for bronze_table, silver_table in target_mappings.items():
    print(f"  {bronze_table:25} → {silver_table}")

# Rename DataFrames in table_dfs using silver target names
print("\nRenaming DataFrames to use silver target table names...")
renamed_dfs = {}
for bronze_name, df in table_dfs.items():
    silver_name = target_mappings.get(bronze_name, bronze_name)
    renamed_dfs[silver_name] = df
    print(f"  {bronze_name:25} → {silver_name}")

# Update table_dfs to use silver names
table_dfs = renamed_dfs

# Create reverse mapping (silver → bronze) for display purposes
silver_to_bronze = {v: k for k, v in target_mappings.items()}

# Update table_names list to use silver names (maintaining same order)
silver_table_names = [target_mappings.get(name, name) for name in table_names]
table_names = silver_table_names

print(f"\nDataFrames now accessible by silver table names: {list(table_dfs.keys())}")
print(f"Table names list updated to: {table_names}")

In [0]:
# Extract schema information (column names and data types) for each table
table_schemas = {}

for table_name, df in table_dfs.items():
    columns = [(field.name, str(field.dataType)) for field in df.schema.fields]
    table_schemas[table_name] = columns
    
    print(f"\n{table_name}:")
    for col_name, col_type in columns:
        print(f"  - {col_name}: {col_type}")

In [0]:
# Step 2b: Identify columns that could serve as join keys and trim string candidates
from pyspark.sql import functions as F

def is_candidate_join_column(col_name):
    """Check if column name suggests it could be a join key"""
    col_lower = col_name.lower()
    patterns = ['id', 'key', 'num', 'nm', 'code', 'number']
    return any(pattern in col_lower for pattern in patterns)

candidate_columns = {}

for table_name, columns in table_schemas.items():
    candidates = [(col_name, col_type) for col_name, col_type in columns 
                  if is_candidate_join_column(col_name)]
    candidate_columns[table_name] = candidates
    
    # Trim string candidates in one pass
    string_candidates = [col for col, dtype in candidates if 'StringType' in dtype]
    if string_candidates:
        table_dfs[table_name] = table_dfs[table_name].withColumns(
            {col: F.trim(F.col(col)) for col in string_candidates}
        )
    
    print(f"\n{table_name}: {len(candidates)} candidate join columns")
    for col_name, col_type in candidates:
        print(f"  - {col_name} ({col_type})")

## Step 3: Display First 100 Rows of Candidate Columns Grouped by Semantic Field

Below we display the first 100 rows of all candidate join columns, organized by their semantic meaning (customer identifiers, product identifiers, etc.). Column names are formatted as `silver_table.column.dtype`.

In [0]:
# Group candidate columns by semantic field and display first 100 rows
from collections import defaultdict
from pyspark.sql import functions as F

def classify_semantic_field(col_name):
    """Classify column into semantic field based on name"""
    col_lower = col_name.lower()
    
    # Customer-related
    if 'cust' in col_lower or 'customer' in col_lower or 'cst' in col_lower:
        return 'Customer Identifiers'
    # Product-related
    elif 'prd' in col_lower or 'product' in col_lower or 'cat' in col_lower or 'category' in col_lower:
        return 'Product Identifiers'
    # Sales/Order-related
    elif 'sale' in col_lower or 'sls' in col_lower or 'order' in col_lower or 'ord' in col_lower:
        return 'Sales/Order Identifiers'
    # Location-related
    elif 'loc' in col_lower or 'location' in col_lower or 'address' in col_lower:
        return 'Location Identifiers'
    # Date-related (sometimes used as keys)
    elif 'date' in col_lower or 'dt' in col_lower:
        return 'Date/Temporal Keys'
    else:
        return 'Other Identifiers'

# Organize by semantic field, preserving table order within each field
semantic_groups = defaultdict(list)

for silver_name in table_names:  # Use locked order (now silver names)
    for col_name, col_type in candidate_columns[silver_name]:
        semantic_field = classify_semantic_field(col_name)
        semantic_groups[semantic_field].append((silver_name, col_name, col_type))

# For each semantic field, build a combined dataframe showing first 100 rows
for semantic_field in sorted(semantic_groups.keys()):
    print(f"\n{'='*80}")
    print(f"Semantic field: {semantic_field}")
    print(f"{'='*80}\n")
    
    # Collect all columns for this semantic field
    columns_in_field = semantic_groups[semantic_field]
    
    if not columns_in_field:
        continue
    
    # Build a combined dataframe with all columns from all tables in this semantic field
    # We'll use UNION to stack rows from different tables
    combined_dfs = []
    
    for silver_name, col_name, col_type in columns_in_field:
        df = table_dfs[silver_name].select(
            F.col(col_name).alias(f"{silver_name}.{col_name}.{col_type}")
        ).limit(100)
        combined_dfs.append(df)
    
    # To display side by side, we need to create a single dataframe
    # We'll use a numbered index approach
    result_df = None
    for i, (silver_name, col_name, col_type) in enumerate(columns_in_field):
        df = table_dfs[silver_name].select(
            F.col(col_name).alias(f"{silver_name}.{col_name}.{col_type}")
        ).limit(100)
        
        # Add row number
        df = df.withColumn("row_num", F.monotonically_increasing_id())
        
        if result_df is None:
            result_df = df
        else:
            # Join on row_num to align rows
            result_df = result_df.join(df, "row_num", "full_outer")
    
    # Drop row_num and display
    if result_df is not None:
        result_df = result_df.drop("row_num")
        display(result_df)

## Step 4: Hypothesis Join Diagram

This diagram proposes join relationships based on schema-level analysis. Each table only shows relationships to tables that appear **below** it in the locked order (to avoid duplication).

**Format:**
* `column_name (primary key) → target_table.target_column ?` = hypothesized relationship
* Indented sub-bullets = suspected data quality issues
* `?` = hypothesis to be tested
* Columns with no arrow = no clear join target identified

### HYPOTHESIS JOIN DIAGRAM

**crm_customers:**
* cst_id (primary key) → crm_sales.sls_cust_id ?
* cst_key → erp_customers.CID ?
  * substring/prefix match suspected (CID = "NAS" + cst_key)
* cst_key → erp_customer_location.CID ?
  * hyphen separator suspected (CID = "AW-00011028" vs cst_key = "AW00011028")

**crm_products:**
* prd_id (primary key)
* prd_key → crm_sales.sls_prd_key ?
  * substring/prefix match suspected (prd_key = "CO-RF-FR-R92R-62" vs sls_prd_key = "BK-R93R-44")
* prd_key → erp_products.ID ?
  * substring/suffix and underscore separator suspected (prd_key = "CO-RF-FR-R92R-62" vs ID = "CO_FO")
* prd_nm

**crm_sales:**
* sls_ord_num (primary key)
* sls_prd_key
* sls_cust_id

**erp_customers:**
* CID → erp_customer_location.CID ?
  * format mismatch suspected (erp_customers.CID = "NASAW00011028" vs erp_customer_location.CID = "AW-00011028")

**erp_customer_location:**
* CID

**erp_products:**
* ID

### Reasoning & Things to Double-Check

**Key observations from the data:**

1. **Customer identifier pattern mismatch:**
   * `crm_customers.cst_key` = "AW00011028" (no separator)
   * `erp_customers.CID` = "NASAW00011028" ("NASAW" prefix added)
   * `erp_customer_location.CID` = "AW-00011028" (hyphen separator added)
   * All three appear to reference the same customer but with different formatting

2. **Product key format mismatch:**
   * `crm_products.prd_key` = "CO-RF-FR-R92R-62" (long hierarchical code)
   * `crm_sales.sls_prd_key` = "BK-R93R-44" (shorter product code)
   * These don't appear to match directly - sls_prd_key may be a different product hierarchy or the actual SKU sold

3. **Integer vs String identifiers:**
   * `crm_customers.cst_id` (integer) vs `cst_key` (string) - same table has both
   * `crm_sales.sls_cust_id` (integer) - different value ranges from cst_id, needs testing

4. **ERP tables use uppercase column names:**
   * `CID`, `ID` vs lowercase in CRM tables

**Testing priorities:**
1. Confirm if `cst_id` = `sls_cust_id` (direct FK) or if transformation is needed
2. Test if `cst_key` substring appears in `erp_cust_az12.CID` and `erp_loc_a101.CID`
3. Investigate product key relationship - may need a bridge table or product hierarchy lookup
4. Check cardinality of all hypothesized joins
5. Measure join coverage (% of records that match vs orphaned records)

## Test Joins

### Define Functions

In [0]:
from pyspark.sql import functions as F

def test_join(source_df, source_col, target_df, target_col, source_name=None, target_name=None, show_unmatched=100):
    """
    Test a join between two columns and report match quality.

    source_df, target_df: DataFrames to join (pre-transformed as needed)
    source_col, target_col: join columns on each side
    source_name, target_name: optional labels for display (defaults to the column name)
    show_unmatched: max number of unmatched rows to display per side (0 to disable)

    Returns:
        dict with test results (source_total, target_total, matched_source,
        matched_target, source_orphan_pct, target_orphan_pct)
    """
    source_name = source_name or source_col
    target_name = target_name or target_col
    join_name = f"{source_name}.{source_col} → {target_name}.{target_col}"

    print("=" * 80)
    print(f"TEST: {join_name}")
    print("=" * 80)

    s = source_df.alias("s")
    t = target_df.alias("t")
    cond = F.col(f"s.{source_col}") == F.col(f"t.{target_col}")

    source_total = source_df.count()
    target_total = target_df.count()

    # Semi-joins avoid row inflation from duplicate keys on either side
    matched_source = s.join(t, cond, "left_semi").count()
    matched_target = t.join(s, cond, "left_semi").count()

    source_orphaned = source_total - matched_source
    target_orphaned = target_total - matched_target
    source_orphan_pct = (source_orphaned / source_total * 100) if source_total > 0 else 0
    target_orphan_pct = (target_orphaned / target_total * 100) if target_total > 0 else 0

    print(f"\nSource ({source_name}.{source_col}):")
    print(f"  Total rows: {source_total:,}")
    print(f"  Matched: {matched_source:,}")
    print(f"  Orphaned: {source_orphaned:,} ({source_orphan_pct:.1f}%)")

    print(f"\nTarget ({target_name}.{target_col}):")
    print(f"  Total rows: {target_total:,}")
    print(f"  Matched: {matched_target:,}")
    print(f"  Orphaned: {target_orphaned:,} ({target_orphan_pct:.1f}%)")

    print(f"\n{'✓ JOIN SUCCESSFUL' if matched_source > 0 else '❌ NO MATCHES FOUND'}")
    print(f"Match quality: {'EXCELLENT' if source_orphan_pct < 1 else 'GOOD' if source_orphan_pct < 10 else 'POOR'}")

    # Display unmatched rows (anti-joins mirror the semi-joins above)
    if show_unmatched > 0:
        if source_orphaned > 0:
            print(f"\n--- Unmatched {source_name} rows (showing up to {show_unmatched} of {source_orphaned:,}) ---")
            s.join(t, cond, "left_anti").limit(show_unmatched).show(show_unmatched, truncate=False)

        if target_orphaned > 0:
            print(f"\n--- Unmatched {target_name} rows (showing up to {show_unmatched} of {target_orphaned:,}) ---")
            t.join(s, cond, "left_anti").limit(show_unmatched).show(show_unmatched, truncate=False)

    return {
        'join_name': join_name,
        'source_total': source_total,
        'target_total': target_total,
        'matched_source': matched_source,
        'matched_target': matched_target,
        'source_orphan_pct': source_orphan_pct,
        'target_orphan_pct': target_orphan_pct,
    }

print("Join test function defined.")

In [0]:
from pyspark.sql import functions as F

def test_substring_uniqueness(df, id_col, start, length=None, new_col_name=None, show_collapsed=0):
    """
    Extract a substring from an id column and check that distinct-value count
    doesn't decrease vs. the full column (i.e. the substring is still unique
    enough to be used as a key).

    df: source DataFrame
    id_col: name of the id column to extract from
    start: substring start position, Spark-style (1-indexed; negative counts
           from the end, e.g. -5 = start at the 5th-from-last character)
    length: how many characters to take, with two conventions:
              - positive int: take exactly that many characters from `start`
              - negative int: take everything from `start` EXCEPT the last
                abs(length) characters (i.e. trim from the end)
              - None: take everything from `start` to the end of the string
    new_col_name: optional name for the extracted column (defaults to
                  f"{id_col}_sub{start}_{length}")
    show_collapsed: if > 0, show up to that many id groups that collapsed onto
                     the same substring value (helps explain a count drop)

    Examples:
        # last 5 characters (suffix)
        test_substring_uniqueness(df, 'CID', start=-5, length=5)

        # everything except the last 5 characters (prefix)
        test_substring_uniqueness(df, 'CID', start=1, length=-5)

    Returns:
        dict with full_count, new_count, diff, and the distinct [id_col, new_col] DataFrame
    """
    col = F.col(id_col)
    total_len = F.length(col)

    if length is None:
        length_expr = -start if start < 0 else total_len - start + 1
    elif length < 0:
        length_expr = total_len + length
    else:
        length_expr = length

    new_col_name = new_col_name or f"{id_col}_sub{start}_{length}"

    print("=" * 80)
    print(f"TEST: uniqueness of {id_col} -> {new_col_name} (start={start}, length={length})")
    print("=" * 80)

    extracted_df = df.select(F.col(id_col), F.substring(col, start, length_expr).alias(new_col_name)).distinct()

    full_count = df.select(id_col).distinct().count()
    new_count = extracted_df.select(new_col_name).distinct().count()
    diff = full_count - new_count

    print(f"\nDistinct ({id_col}):        {full_count:,}")
    print(f"Distinct ({new_col_name}):  {new_count:,}")
    print(f"Difference:                 {diff:,}")
    print(f"\n{'✓ UNIQUENESS PRESERVED' if diff == 0 else '❌ UNIQUENESS LOST — collision(s) introduced'}")

    if show_collapsed > 0 and diff > 0:
        print(f"\n--- Collapsed groups (showing up to {show_collapsed}) ---")
        (
            extracted_df.groupBy(new_col_name)
            .agg(F.count(id_col).alias("id_count"))
            .filter(F.col("id_count") > 1)
            .orderBy(F.desc("id_count"))
            .limit(show_collapsed)
            .show(show_collapsed, truncate=False)
        )

    return {
        'id_col': id_col,
        'new_col_name': new_col_name,
        'full_count': full_count,
        'new_count': new_count,
        'diff': diff,
        'df': extracted_df,
    }

print("Substring uniqueness test function defined.")

In [0]:
from pyspark.sql import functions as F

def extract_by_segment(col, delimiter='-', drop_first=0, drop_last=0):
    """
    Split a column on `delimiter` and rejoin all segments except the first
    `drop_first` and last `drop_last`, however many segments there are.

    Useful where fixed-width substring slicing breaks down because segment
    lengths vary but the *number* of leading/trailing segments to drop
    is constant. E.g. 'CO-MF-FR-M21B-40' with drop_first=2 -> 'FR-M21B-40'.

    col: Spark Column to split (e.g. F.col('CID'))
    delimiter: literal string to split on (treated as regex by F.split —
               escape it yourself if it contains regex metacharacters)
    drop_first: number of leading segments to drop
    drop_last: number of trailing segments to drop
    """
    parts = F.split(col, delimiter)
    n_parts = F.size(parts)
    start = drop_first + 1                       # F.slice is 1-indexed
    length = n_parts - drop_first - drop_last
    return F.array_join(F.slice(parts, start, length), delimiter)


def test_segment_uniqueness(df, id_col, delimiter='-', drop_first=0, drop_last=0, new_col_name=None, show_collapsed=0):
    """
    Extract a segment-based substring from an id column (via extract_by_segment)
    and check that distinct-value count doesn't decrease vs. the full column.

    df: source DataFrame
    id_col: name of the id column to extract from
    delimiter: literal string to split id_col on
    drop_first: number of leading segments to drop
    drop_last: number of trailing segments to drop
    new_col_name: optional name for the extracted column (defaults to
                  f"{id_col}_seg_d{drop_first}_{drop_last}")
    show_collapsed: if > 0, show up to that many id groups that collapsed onto
                     the same substring value (helps explain a count drop)

    Returns:
        dict with full_count, new_count, diff, and the distinct [id_col, new_col] DataFrame
    """
    col = F.col(id_col)
    new_col_name = new_col_name or f"{id_col}_seg_d{drop_first}_{drop_last}"
    expr = extract_by_segment(col, delimiter=delimiter, drop_first=drop_first, drop_last=drop_last)

    print("=" * 80)
    print(f"TEST: uniqueness of {id_col} -> {new_col_name} (delimiter='{delimiter}', drop_first={drop_first}, drop_last={drop_last})")
    print("=" * 80)

    extracted_df = df.select(col, expr.alias(new_col_name)).distinct()

    full_count = df.select(id_col).distinct().count()
    new_count = extracted_df.select(new_col_name).distinct().count()
    diff = full_count - new_count

    print(f"\nDistinct ({id_col}):        {full_count:,}")
    print(f"Distinct ({new_col_name}):  {new_count:,}")
    print(f"New column name:            {new_col_name}")
    print(f"Difference:                 {diff:,}")
    print(f"\n{'✓ UNIQUENESS PRESERVED' if diff == 0 else '❌ UNIQUENESS LOST — collision(s) introduced'}")

    if show_collapsed > 0 and diff > 0:
        print(f"\n--- Collapsed groups (showing up to {show_collapsed}) ---")
        (
            extracted_df.groupBy(new_col_name)
            .agg(F.count(id_col).alias("id_count"))
            .filter(F.col("id_count") > 1)
            .orderBy(F.desc("id_count"))
            .limit(show_collapsed)
            .show(show_collapsed, truncate=False)
        )

    return {
        'id_col': id_col,
        'new_col_name': new_col_name,
        'full_count': full_count,
        'new_count': new_count,
        'diff': diff,
        'df': extracted_df,
    }

print("Segment extraction and uniqueness test functions defined.")

### Test 1: crm_customers.cst_id → crm_sales.sls_cust_id

In [0]:
# transform freely first
crm_customers = table_dfs['crm_customers']
crm_sales = table_dfs['crm_sales']

result_1 = test_join(
    crm_customers, 'cst_id',
    crm_sales, 'sls_cust_id',
    source_name='crm_customers', target_name='crm_sales'
)

#### Test 1: RESULT
RELATIONSHIP CONFIRMED: crm_customers.cst_id → crm_sales.sls_cust_id

ACTION:  
No action required.

### Test 2: crm_customers.cst_key → erp_customers.CID (join on last 5 characters)


In [0]:
# Verify: cst_id is always in the last 5 characters of cst_key
from pyspark.sql import functions as F

print("="*80)
print("VERIFICATION: cst_id embedded in cst_key")
print("="*80)

crm_customers_df = table_dfs['crm_customers']

# Extract last 5 characters of cst_key and compare to cst_id (as string)
verification_df = crm_customers_df.withColumn(
    'cst_key_suffix', F.substring(F.col('cst_key'), -5, 5)
).withColumn(
    'cst_id_str', F.col('cst_id').cast('string')
).withColumn(
    'match', F.col('cst_key_suffix') == F.col('cst_id_str')
)

# Count matches vs mismatches
total_rows = verification_df.count()
matches = verification_df.filter(F.col('match') == True).count()
mismatches = verification_df.filter(F.col('match') == False).count()

print(f"\nTotal rows: {total_rows:,}")
print(f"Matches: {matches:,}")
print(f"Mismatches: {mismatches:,}")

if mismatches == 0:
    print("\n✓ CONFIRMED: cst_id always appears in the last 5 characters of cst_key")
else:
    print(f"\n❌ NOT CONFIRMED: {mismatches} rows where cst_id does not match")
    print("\nSample mismatches:")
    verification_df.filter(F.col('match') == False).select(
        'cst_id', 'cst_id_str', 'cst_key', 'cst_key_suffix'
    ).show(10, truncate=False)

In [0]:
cst_key_suffix = test_substring_uniqueness(
    table_dfs['crm_customers'], 'cst_key', start=-5, length=5
)

source = cst_key_suffix['df']

CID_suffix = test_substring_uniqueness(
    table_dfs['erp_customers'], 'CID', start=-5, length=5
)

target = CID_suffix['df']

In [0]:
result_2 = test_join(
    source, 'cst_key_sub-5_5',
    target, 'CID_sub-5_5',
    source_name='crm_customers', target_name='erp_customers'
)

#### Test 2: RESULT
RELATIONSHIP CONFIRMED. crm_customers.cst_key and erp_customers.CID both contain crm_customers.cst_id as a substring in the last 5 characters. Further more the prefixs in each row do not contribute at all to the uniqueness of the columns. 4 Orphand rows are due to bad inputs.

ACTION: Both crm_customers.cst_key and erp_customers.CID should be split into two columns in their respective silver notebooks, one id column for the last 5 characters and the other column should record the prefixes. The column name for both of the prefix columns should be category_id.  
Previously it was shown that the last 5 characters of crm_customers.cst_key is an exact match of crm_customers.cst_id except in 4 cases where crm_customers.cst_id is null. But since these 4 cases clearly have nonsens cst_ids (see Test 1):  
|A01Ass  |  
|SF566   |  
|PO25    |  
|13451235|  
Then we can drop the split suffix column from crm_customers.cst_key and this will not remove any important information.  

The suffix column from erp_customers.CID should be renamed to customer_id.

### Test 3: crm_customers.cst_key → erp_customer_location.CID (join on last 5 characters)
It appears that erp_customer_location.CID also contains cst_id in the last 5 characters.

Use the same approach as crm_customers.cst_key → erp_customers.CID relationship

In [0]:
CID_suffix = test_substring_uniqueness(
    table_dfs['erp_customer_location'], 'CID', start=-5, length=5
)

target = CID_suffix['df']

In [0]:
# Use same source table from previous test
result_3 = test_join(
    source, 'cst_key_sub-5_5',
    target, 'CID_sub-5_5',
    source_name='crm_customers', target_name='erp_customer_location'
)

#### Test 3: RESULT
RELATIONSHIP CONFIRMED (Same as result 2). crm_customers.cst_key and erp_customer_location.CID both contain crm_customers.cst_id as a substring in the last 5 characters. Further more the prefixs in each row do not contribute at all to the uniqueness of the columns. 4 Orphand rows are due to bad inputs.

ACTION: Both crm_customers.cst_key and erp_customer_location.CID should be split into two columns in their respective silver notebooks, one id column for the last 5 characters and the other column should record the prefixes.
The column name for both of the prefix columns should be category_id.
Previously it was shown that the last 5 characters of crm_customers.cst_key is an exact match of crm_customers.cst_id except in 4 cases where crm_customers.cst_id is null. But since these 4 cases clearly have nonsens cst_ids (see Test 1):   
|A01Ass  |  
|SF566   |  
|PO25    |  
|13451235|  
Then we can drop the split suffix column from crm_customers.cst_key and this will not remove any important information.

The suffix column from erp_customer_location.CID should be renamed to customer_id.

### Test 4: crm_products.prd_key → crm_sales.sls_prd_key
Suspected that crm_sales.sls_prd_key is contained in the last 10 characters of crm_products.prd_key.

In [0]:
prd_key_segment = test_segment_uniqueness(
    table_dfs['crm_products'],
    id_col='prd_key',
    delimiter='-',
    drop_first=2,
    show_collapsed=20,
)

source = prd_key_segment['df']

In [0]:
target = table_dfs['crm_sales']

result_4 = test_join(
    source, 'prd_key_seg_d2_0',
    target, 'sls_prd_key',
    source_name='crm_products', target_name='crm_sales'
)

#### Test 4: RESULT
RELATIONSHIP CONFIRMED. crm_products.prd_key → crm_sales.sls_prd_key when you separate crm_products.prd_key at the second segemnt denoted by '-' and take the right side. All sales get associated with a product but some products were not sold which contributes to a high orphan rate.

ACTION: Split crm_products.prd_key in the same way we have done in this join test by separating on the second segment ('-'). 
Examples:  
|CO-MF-FR-M21B-40| -> |CO-MF|FR-M21B-40|  
|CL-GL-GL-F110-L| -> |CL-GL|GL-F110-L|  
|AC-BC-WB-H098| -> |AC-BC|WB-H098|  
The left hand side of the split e.g. CO-MF should be called category_id and the right hang side e.g. FR-M21B-40 should be called product_key since it is what we use to join to sales. 

### Test 5: crm_products.prd_key → erp_products.ID
substring/suffix and underscore separator suspected (prd_key = "CO-RF-FR-R92R-62" vs ID = "CO_FO")

In [0]:
prd_key_prefix = test_substring_uniqueness(
    table_dfs['crm_products'], 'prd_key', start=1, length=5
)

source = prd_key_prefix['df']
source.display()

target = table_dfs['erp_products']
# Transform erp_products.ID
target = target.withColumn('ID', F.regexp_replace('ID', '_', '-'))
target.display()

We can see that taking the first 5 characters of crm_products.prd_key does not preserve the uniqueness of the column reducing the number of unique ids down to 37. However the target column erp_products.ID also has 37 id's. 

In [0]:
result_5 = test_join(
    source, 'prd_key_sub1_5',
    target, 'ID',
    source_name='crm_products', target_name='erp_products'
)

####Test 5: RESULT  
RELATIONSHIP CONFIRMED. crm_products.prd_key → erp_products.ID when you take the first 5 characters from crm_products.prd_key and replace '_' with '-' in erp_products.ID. Only 1 product does not get any matches which may be due to a input error.  

ACTION: replace replace '_' with '-' in erp_products.ID and rename it to category_id. crm_products.prd_key has already been split into the correct columns according to the results in the previous test. 

### Test 6: erp_customers.CID → erp_customer_location.CID ?
format mismatch suspected (erp_customers.CID = "NASAW00011028" vs erp_customer_location.CID = "AW-00011028")
These two columns have alreaady been split into category_id and customer_id parts according to results from previous tests.

In [0]:
cst_key_suffix = test_substring_uniqueness(
    table_dfs['erp_customers'], 'CID', start=-5, length=5
)

source = cst_key_suffix['df']

CID_suffix = test_substring_uniqueness(
    table_dfs['erp_customer_location'], 'CID', start=-5, length=5
)

target = CID_suffix['df']

In [0]:
result_6 = test_join(
    source, 'CID_sub-5_5',
    target, 'CID_sub-5_5',
    source_name='erp_customers', target_name='erp_customer_location'
)

#### Test 6: RESULT
RELATIONSHIP CONFIRMED: erp_customers.CID → erp_customer_location.CID after taking the last 5 character substrings from each

ACTION:
No action required. These actions have already been completed during previous steps.

## Step 6: Test Results Diagram

Structure matches the hypothesis diagram, with tick (✓) for at least one match, and orphan percentages shown.

### TEST RESULTS DIAGRAM

**crm_customers:**
* cst_id (primary key) → crm_sales.sls_cust_id **✓** (0.0% source orphaned, 0.0% target orphaned)
  * Direct integer join confirmed
  * Match quality: EXCELLENT
* cst_key (last 5 chars) → erp_customers.CID (last 5 chars) **✓** (0.0% source orphaned, 0.0% target orphaned)
  * Substring match on last 5 characters confirmed
  * Match quality: EXCELLENT
  * 4 orphaned rows due to invalid source data (NULL cst_id)
* cst_key (last 5 chars) → erp_customer_location.CID (last 5 chars) **✓** (0.0% source orphaned, 0.0% target orphaned)
  * Substring match on last 5 characters confirmed
  * Match quality: EXCELLENT
  * Same 4 orphaned rows as above

**crm_products:**
* prd_id (primary key)
* prd_key (drop first 2 segments) → crm_sales.sls_prd_key **✓** (55.9% source orphaned, 0.0% target orphaned)
  * All sales records matched to products
  * 165 of 295 products have no sales (normal business scenario)
  * Match quality: POOR (due to high product orphan rate)
* prd_key (first 5 chars) → erp_products.ID **✓** (2.4% source orphaned, 2.7% target orphaned)
  * Category-level join confirmed (with "-" → "_" replacement in erp_products.ID)
  * Match quality: GOOD
  * 7 source products with category "CO-PE" (Pedals) orphaned
  * 1 target category "CO-PD" orphaned (likely data entry error: PD vs PE)
* prd_nm

**crm_sales:**
* sls_ord_num (primary key)
* sls_prd_key
* sls_cust_id

**erp_customers:**
* CID (last 5 chars) → erp_customer_location.CID (last 5 chars) **✓** (0.0% source orphaned, 0.0% target orphaned)
  * Substring match on last 5 characters (customer ID portion) confirmed
  * Match quality: EXCELLENT
  * Perfect match after extracting customer ID from both formatted keys

**erp_customer_location:**
* CID

**erp_products:**
* ID

## Step 7: Transformation Table

### Successfully Resolved Transformations

| Source Table | Source Column | Target Table | Target Column | Transformation | Source Orphan | Target Orphan | Match Quality |
|---|---|---|---|---|---|---|---|
| crm_customers | cst_id | crm_sales | sls_cust_id | None (direct match) | 0.0% | 0.0% | EXCELLENT |
| crm_customers | cst_key | erp_customers | CID | Extract last 5 chars from both: `SUBSTRING(col, -5, 5)` | 0.0% | 0.0% | EXCELLENT |
| crm_customers | cst_key | erp_customer_location | CID | Extract last 5 chars from both: `SUBSTRING(col, -5, 5)` | 0.0% | 0.0% | EXCELLENT |
| crm_products | prd_key | erp_products | ID | Extract first 5 chars from prd_key, replace "_" with "-" in ID | 2.4% | 2.7% | GOOD |
| crm_products | prd_key | crm_sales | sls_prd_key | Drop first 2 segments (split on '-'): `array_join(slice(split(prd_key, '-'), 3, size-2), '-')` | 55.9% | 0.0% | POOR |
| erp_customers | CID | erp_customer_location | CID | Extract last 5 chars from both: `SUBSTRING(col, -5, 5)` | 0.0% | 0.0% | EXCELLENT |

### Notes on Transformations

1. **Customer ID Relationships (0.0% orphans)**
   * All customer identifier columns (cst_key, CID in both ERP tables) contain the customer ID in their last 5 characters
   * The 4 orphaned records in crm_customers are data quality issues (NULL cst_id with invalid cst_key values)
   * Recommendation: Extract last 5 characters as `customer_id` in all silver tables

2. **Product Category Match (2.4-2.7% orphans)**
   * crm_products.prd_key first 5 chars = product category (e.g., "CO-MF")
   * erp_products.ID uses underscore separator (e.g., "CO_MF") for the same categories
   * 7 products with "CO-PE" category orphaned, 1 target "CO-PD" orphaned (data entry error: Pedals vs PD)
   * Recommendation: Standardize on hyphen separator and extract as `category_id`

3. **Product-to-Sales Match (55.9% product orphans, all sales matched)**
   * All 60,398 sales records successfully match to products
   * 165 of 295 products have no sales - this is normal (not all products sell)
   * The transformation (drop first 2 segments) successfully creates the join key
   * Recommendation: This is production-ready; high orphan rate is expected business behavior

## Summary & Next Steps

### Key Findings

**All 6 joins successfully validated with transformations:**

1. **crm_customers.cst_id → crm_sales.sls_cust_id** (0.0% orphaned)
   * Direct match, production ready
   
2. **crm_customers.cst_key → erp_customers.CID** (0.0% orphaned)
   * Extract last 5 characters from both
   * Production ready
   
3. **crm_customers.cst_key → erp_customer_location.CID** (0.0% orphaned)
   * Extract last 5 characters from both
   * Production ready
   
4. **crm_products.prd_key → crm_sales.sls_prd_key** (55.9% source orphaned, 0.0% target orphaned)
   * Drop first 2 segments from prd_key
   * All sales matched to products
   * High product orphan rate is expected (products with no sales)
   * Production ready
   
5. **crm_products.prd_key → erp_products.ID** (2.4% source orphaned, 2.7% target orphaned)
   * Extract first 5 chars from prd_key, replace "_" with "-" in ID
   * Nearly perfect match; orphans due to data entry error ("CO-PE" vs "CO-PD")
   * Production ready
   
6. **erp_customers.CID → erp_customer_location.CID** (0.0% orphaned)
   * Extract last 5 characters from both
   * Production ready

### Recommended Column Transformations for Silver Layer

Based on test results, apply these transformations:

**1. crm_customers:**
* Extract `customer_id`: `SUBSTRING(cst_key, -5, 5)` (last 5 chars)
* Extract `category_id`: `SUBSTRING(cst_key, 1, LENGTH(cst_key)-5)` (everything except last 5)

**2. crm_products:**
* Extract `category_id`: `SUBSTRING(prd_key, 1, 5)` (first 5 chars, e.g. "CO-MF")
* Extract `product_key`: Drop first 2 segments: `array_join(slice(split(prd_key, '-'), 3, size(split(prd_key, '-'))-2), '-')`
  * Example: "CO-MF-FR-M21B-40" → "FR-M21B-40"

**3. erp_customers:**
* Extract `customer_id`: `SUBSTRING(CID, -5, 5)` (last 5 chars)
* Extract `category_id`: `SUBSTRING(CID, 1, LENGTH(CID)-5)` (everything except last 5)

**4. erp_customer_location:**
* Extract `customer_id`: `SUBSTRING(CID, -5, 5)` (last 5 chars)
* Extract `category_id`: `SUBSTRING(CID, 1, LENGTH(CID)-5)` (everything except last 5)

**5. erp_products:**
* Standardize `category_id`: `REGEXP_REPLACE(ID, '_', '-')` (replace underscore with hyphen)

### Data Quality Issues Identified

* **4 invalid customer records** in crm_customers with NULL cst_id and malformed cst_key values:
  * A01Ass, SF566, PO25, 13451235
  * Recommendation: Filter these out in silver layer
  
* **7 products with "CO-PE" category** have no matching erp_products record
  * 1 erp_products record with "CO-PD" category has no matching crm_products
  * Likely data entry error: "PE" vs "PD" for Pedals category
  * Recommendation: Investigate and standardize

### Next Steps

1. **Implement transformations** in silver layer notebooks using the exact SQL expressions above
2. **Add join keys** as new columns: `customer_id`, `category_id`, `product_key`
3. **Filter invalid data** (4 customer records with NULL cst_id)
4. **Investigate data entry error** for Pedals category (PE vs PD)
5. **Validate joins** in silver layer after transformations
6. **Document relationships** in data dictionary for downstream users